# ch14 练习 —— 重塑与透视

> 数据：`device_defects.csv`（270 x 13，18 台区 × 5 设备类型）
> 已备好：`long`（88 个组合的双层索引）、`wide`（18 x 5 宽表）

**做题规则**：从上往下依次填，每个 `____` 处跑通再往下。
验收块 `assert` 不要改，它是你自己对账用的。

| 题号 | 考点 |
|---|---|
| TODO(1) | `crosstab` 计数表与 `margins` |
| TODO(2) | `crosstab` 的 `normalize` 与 `values+aggfunc` |
| TODO(3) | `pivot_table` 与 `groupby+unstack` 的等价 |
| TODO(4) | 重复索引下 `pivot` / `unstack` 报错 |
| TODO(5) | 宽表 `NaN` 的两张面孔（`fill_value` 诊断） |
| TODO(6) | `unstack` 的 `level` 方向 + 两级列名压平 |
| TODO(7) | `stack` 的往返与 3.0 的两处变更 |
| TODO(8) | `melt` 宽 → 长 |
| TODO(9) | `explode` 的正确顺序 |
| TODO(10) | `explode` 的多分隔符与索引 |
| 综合① | 长 ↔ 宽 闭环 + 行数对账 |
| 综合② | 可交付交叉报表 |

共 12 个挖空块。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

dev = pd.read_csv(DATA / "device_defects.csv")

ROW_KEY = "台区编号"
COL_KEY = "设备类型"
LIST_KEY = "缺陷等级"
VALUE_COL = "负荷值"

# 长表（Series 形态）：每个 (台区编号, 设备类型) 组合一行 —— 88 个组合
long = dev.groupby([ROW_KEY, COL_KEY])[VALUE_COL].mean()

# 宽表：把 设备类型 展开成列 —— (18, 5)
wide = long.unstack(COL_KEY)

# 对照用的 DataFrame 形态长表：unstack 后列名会是两级的（详见讲解 §3.2）
long_df = (
    dev.groupby([ROW_KEY, COL_KEY], as_index=False)[VALUE_COL]
    .mean()
    .set_index([ROW_KEY, COL_KEY])
)

print("pandas", pd.__version__)
print("dev", dev.shape, "| long", long.shape, "| wide", wide.shape)
print("wide 列名:", list(wide.columns), "| 列轴名字:", wide.columns.name)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def grid_to_long(frame, row_key, col_key, value_name):
    """宽 -> 长：把列名收成一列（列名来源放进 col_key）。"""
    return (
        frame.reset_index()
        .melt(id_vars=row_key, var_name=col_key, value_name=value_name)
    )


def nan_audit(frame):
    """宽表的 NaN 有多少个，以及每行/每列各有多少。"""
    return {
        "总数": int(frame.isna().sum().sum()),
        "按行": frame.isna().sum(axis=1).to_dict(),
    }


print("long 组合数:", len(long), "| 宽表满格数:", wide.shape[0] * wide.shape[1])

---

## 基础题

In [ ]:
# TODO(1)：用 crosstab 造「设备类型 x 缺陷等级」计数表存入 ct；
#           再带 margins=True 存入 ct_margin；两者总和分别存入 n_ct / n_margin
# 提示：pd.crosstab(dev[COL_KEY], dev[LIST_KEY]) ；加 margins=True
#        总和用 ct.to_numpy().sum()
ct = ____
ct_margin = ____
n_ct = ____
n_margin = ____
# ---- 验收 ----
assert ct.shape == (5, 3), f"计数表应为 (5, 3)，实际 {ct.shape}"
assert n_ct == 270, f"计数表总和应等于 dev 行数 270，实际 {n_ct}"
assert ct_margin.shape == (6, 4), f"带 margins 应为 (6, 4)，实际 {ct_margin.shape}"
assert n_margin == n_ct + int(ct.sum(axis=1).sum()) + int(ct.sum(axis=0).sum()) + 270, \
    "margins 表的总和 = 原表 + 行合计 + 列合计 + 总计"
assert int(ct_margin.loc["All", "All"]) == 270, "右下角总计应为 270"
assert ct.sum(axis=1).to_dict() == {"互感器": 58, "变压器": 55, "断路器": 41, "绝缘子": 57, "避雷器": 59}, \
    f"行合计不对：{ct.sum(axis=1).to_dict()}"
assert ct.sum(axis=0).to_dict() == {"一般": 164, "严重": 75, "危急": 31}, \
    f"列合计不对：{ct.sum(axis=0).to_dict()}"
print("TODO(1) 通过 ->", ct.shape, "总和", n_ct, "| margins", ct_margin.shape)

In [ ]:
# TODO(2)：行归一表存入 ct_norm（每个设备类型的等级占比）；
#           再把 values 设为负荷值、aggfunc 用 mean，存入 ct_mean
# 提示：pd.crosstab(dev[COL_KEY], dev[LIST_KEY], normalize="index")
#        pd.crosstab(dev[COL_KEY], dev[LIST_KEY], values=dev[VALUE_COL], aggfunc="mean")
ct_norm = ____
ct_mean = ____
# ---- 验收 ----
assert ct_norm.shape == (5, 3), f"归一表形状应为 (5, 3)，实际 {ct_norm.shape}"
assert np.allclose(ct_norm.sum(axis=1).to_numpy(), 1.0), "行归一的每行之和应为 1"
assert int((ct_norm > 0).to_numpy().sum()) == 15, "本数据 5x3 共 15 格都有记录，归一后全部非零"
assert abs(float(ct_norm.loc["避雷器", "危急"]) - 13 / 59) < 1e-9, \
    "避雷器×危急 的占比应为 13/59"
assert int(ct_mean.isna().sum().sum()) == 0, "这份数据每个组合都有值，不该有 NaN"
assert abs(float(ct_mean.loc["变压器", "一般"]) - 1311.664) < 0.01, \
    f"变压器×一般的均值应为 1311.664，实际 {float(ct_mean.loc['变压器', '一般']):.3f}"
assert abs(float(ct_mean.loc["避雷器", "一般"]) - 39.039) < 0.01, "避雷器×一般应为 39.039"
print("TODO(2) 通过 -> 归一表行和全 1 | 聚合表 NaN 0 | 变压器×一般 =",
      round(float(ct_mean.loc["变压器", "一般"]), 3))

In [ ]:
# TODO(3)：验证 pivot_table 与 groupby+unstack 的等价：
#           pivot_table 结果存入 via_pt（index=ROW_KEY, columns=COL_KEY, values=负荷值），
#           groupby+unstack 结果存入 via_gb；两者形状存入 shape_pt / shape_gb
# 提示：dev.pivot_table(index=ROW_KEY, columns=COL_KEY, values=VALUE_COL)
#        dev.groupby([ROW_KEY, COL_KEY])[VALUE_COL].mean().unstack(COL_KEY)
via_pt = ____
via_gb = ____
shape_pt = ____
shape_gb = ____
# ---- 验收 ----
assert shape_pt == (18, 5), f"pivot_table 应为 (18, 5)，实际 {shape_pt}"
assert shape_gb == (18, 5), f"groupby+unstack 应为 (18, 5)，实际 {shape_gb}"
assert list(via_pt.columns) == list(via_gb.columns), "两种写法的列名顺序应一致"
assert int(via_pt.isna().sum().sum()) == 3, f"两种写法都应有 3 个 NaN，实际 {int(via_pt.isna().sum().sum())}"
max_diff = float((via_pt - via_gb).abs().max().max())
assert max_diff < 1e-9, f"非空格子数值应完全一致，实际最大差值 {max_diff}"
assert via_gb.columns.nlevels == 1, "Series.unstack 的列名是单级的"
print("TODO(3) 通过 -> pivot_table 与 groupby+unstack 完全等价，NaN 均为 3")

In [ ]:
# TODO(4)：用 probe 探测重复索引下的报错：
#           dev.pivot(index=ROW_KEY, columns=COL_KEY, values=VALUE_COL) → probe_pivot
#           dev.set_index([ROW_KEY,COL_KEY])[[VALUE_COL]].unstack(1) → probe_unstack
#           dev.pivot_table(...) 的形状 → shape_pt_ok
# 提示：probe(dev.pivot, index=ROW_KEY, columns=COL_KEY, values=VALUE_COL)
probe_pivot = ____
probe_unstack = ____
shape_pt_ok = ____
# ---- 验收 ----
assert probe_pivot[0] == "err" and probe_pivot[1] == "ValueError", \
    f"pivot 遇重复索引应抛 ValueError，实际 {probe_pivot}"
assert probe_unstack[0] == "err" and probe_unstack[1] == "ValueError", \
    f"unstack 遇重复索引应抛 ValueError，实际 {probe_unstack}"
assert shape_pt_ok == (18, 5), "pivot_table 能聚合，应正常返回"
dup = dev.set_index([ROW_KEY, COL_KEY])
assert dup.index.has_duplicates is True, "270 行只对应 88 个唯一组合"
assert dup.index.nunique() == 88, f"唯一组合应为 88，实际 {dup.index.nunique()}"
print("TODO(4) 通过 -> pivot / unstack 均抛", probe_pivot[1], "| pivot_table 正常", shape_pt_ok)

In [ ]:
# TODO(5)：诊断宽表 NaN 的两张面孔：
#           wide_raw = long.unstack(1)（NaN 总数存入 n_raw）
#           wide_filled = long.unstack(1, fill_value=-1)（剩余 NaN 存入 n_left，-1 格数存入 n_filled）
#           再定位「值本身缺失」的格子，把它的元组存入 cell_missing
# 提示：long.unstack(1, fill_value=-1) ；真正全缺的格子填 -1 之后仍是 NaN
wide_raw = ____
n_raw = ____
wide_filled = ____
n_left = ____
n_filled = ____
still_nan = ____
cell_missing = ____
# ---- 验收 ----
assert n_raw == 3, f"原始宽表应有 3 个 NaN，实际 {n_raw}"
assert n_filled == 2, f"「组合不存在」的格子应为 2 个，实际 {n_filled}"
assert n_left == 1, f"「值本身缺失」的格子应为 1 个，实际 {n_left}"
assert n_filled + n_left == n_raw, "3 = 2（组合不存在）+ 1（值本身缺失）"
assert cell_missing == ("STATION_A_01", "变压器"), f"该格子应为 (STATION_A_01, 变压器)，实际 {cell_missing}"
# 该组合确实存在，但负荷值全缺
subset = dev[(dev[ROW_KEY] == "STATION_A_01") & (dev[COL_KEY] == "变压器")]
assert len(subset) > 0, "该组合在明细里确实存在"
assert int(subset[VALUE_COL].isna().sum()) == len(subset), "该组合的负荷值应全为缺失"
print("TODO(5) 通过 -> 3 = 2（组合不存在）+ 1（值本身缺失）")
print("   填不掉的格子:", cell_missing, "| 该组合明细", len(subset), "行，负荷值全缺")

In [ ]:
# TODO(6)：验证 unstack 的 level 方向：
#           unstack() 默认结果形状存入 shape_default
#           unstack(1) 形状存入 shape_lv1，unstack(0) 形状存入 shape_lv0
#           unstack(0) 的索引层级名存入 idx_lv0
# 提示：long.unstack() ；long.unstack(1) ；long.unstack(0) ；.index.names
shape_default = ____
shape_lv1 = ____
res_lv0 = ____
shape_lv0 = ____
idx_lv0 = ____
# ---- 验收 ----
assert shape_default == (18, 5), f"默认（拆最后一层）应为 (18, 5)，实际 {shape_default}"
assert shape_lv1 == (18, 5), "unstack(1) 应与默认相同"
assert shape_lv0 == (5, 18), f"unstack(0) 应把台区摊成列，实际 {shape_lv0}"
assert idx_lv0 == [COL_KEY], f"unstack(0) 的索引应是设备类型，实际 {idx_lv0}"
assert shape_lv0 == (shape_lv1[1], shape_lv1[0]), "两个方向的形状应互为转置"
assert long.index.names == [ROW_KEY, COL_KEY], f"原始索引层级：{long.index.names}"
# 附带验证：DataFrame 形态长表 unstack 出两级列名，Series 形态不会
assert long_df.unstack(1).columns.nlevels == 2, "DataFrame.unstack 应产生两级列名"
assert long.unstack(COL_KEY).columns.nlevels == 1, "Series.unstack 的列名是单级的"
assert wide.columns.name == COL_KEY, "Series.unstack 会给列轴起上原名"
assert probe(lambda: long_df.unstack(1)["互感器"])[1] == "KeyError", \
    "两级列名下用单层键取列会 KeyError —— 所以长表要用 Series 形态"
print("TODO(6) 通过 -> 默认", shape_default, "| unstack(1)", shape_lv1,
      "| unstack(0)", shape_lv0, "索引", idx_lv0)
print("   DataFrame 形态列名层级 ->", long_df.unstack(1).columns.nlevels,
      "| Series 形态 ->", long.unstack(COL_KEY).columns.nlevels)

---

## 进阶题

In [ ]:
# TODO(7)：stack 的往返与 3.0 两处变更：
#           wide.stack() 的形状存入 shape_stack，索引层级存入 idx_stack，NaN 数存入 n_stack_nan
#           dropna=False 的探测结果存入 probe_dropna
#           Series 是否还有 stack 方法存入 has_series_stack
# 提示：wide.stack() ；probe(wide.stack, dropna=False) ；hasattr(wide[COL_KEY], "stack")
stacked = ____
shape_stack = ____
idx_stack = ____
n_stack_nan = ____
probe_dropna = ____
has_series_stack = ____
# ---- 验收 ----
assert shape_stack == (90,), f"stack 后应为 90 行（18 x 5 格），实际 {shape_stack}"
assert idx_stack == [ROW_KEY, COL_KEY], f"stack 后索引层级应为 [台区编号, 设备类型]，实际 {idx_stack}"
assert n_stack_nan == 3, f"3.0 的新 stack 不丢 NaN，应保留 3 个，实际 {n_stack_nan}"
assert probe_dropna[0] == "err", "pandas 3.0 的 stack 不接受 dropna 参数"
assert probe_dropna[1] == "ValueError", f"应抛 ValueError，实际 {probe_dropna[1]}"
assert has_series_stack is False, "pandas 3.0 已移除 Series.stack"
assert probe(wide.stack, future_stack=True)[0] == "ok", "future_stack=True 仍可用"
print("TODO(7) 通过 -> stack", shape_stack, "索引", idx_stack, "| NaN", n_stack_nan,
      "| dropna 报", probe_dropna[1], "| Series.stack 存在:", has_series_stack)

In [ ]:
# TODO(8)：用 melt 把 wide 变回长表：
#           先把列轴命名为 COL_KEY，再 reset_index 后 melt(id_vars=ROW_KEY,
#           var_name=COL_KEY, value_name="平均负荷")，结果存入 long_back
#           形状存入 shape_melt，NaN 数存入 n_melt_nan
# 提示：wide.columns.name = COL_KEY ；wide.reset_index().melt(...)
wide_tbl = ____
wide_tbl.columns.name = ____
long_back = ____
shape_melt = ____
n_melt_nan = ____
# ---- 验收 ----
assert shape_melt == (90, 3), f"melt 后应为 (90, 3)（满格数），实际 {shape_melt}"
assert list(long_back.columns) == [ROW_KEY, COL_KEY, "平均负荷"], \
    f"列名应为 [台区编号, 设备类型, 平均负荷]，实际 {list(long_back.columns)}"
assert isinstance(long_back.index, pd.RangeIndex), "melt 产出 RangeIndex，索引干净"
assert n_melt_nan == 3, f"melt 带出 3 个空格子，实际 {n_melt_nan}"
assert abs(float(long_back["平均负荷"].sum()) - float(np.nansum(wide.to_numpy()))) < 1e-9, \
    "melt 后求和应与宽表 nan 求和一致"
# 关键：90 ≠ 88，必须显式 dropna 才能回到组合数
assert len(long_back) == 90 and len(long) == 88, "满格 90 vs 组合 88，两者不相等"
print("TODO(8) 通过 -> melt", shape_melt, "| NaN", n_melt_nan,
      "| 90 格 vs 88 组合（长→宽→长 行数不守恒）")

In [ ]:
# TODO(9)：explode 的正确顺序：
#           造 split_demo = {编号:[1,2,3,4], 缺陷类型:["渗油/异物","放电,发热","无","发热/放电/无"]}
#           直接 explode 的行数存入 n_wrong
#           先 str.split("/") 再 explode 的结果存入 right，行数存入 n_right
# 提示：split_demo.explode("缺陷类型") ；split_demo["缺陷类型"].str.split("/").explode()
split_demo = ____
n_wrong = ____
right = ____
n_right = ____
# ---- 验收 ----
assert n_wrong == 4, f"不先 split 时 explode 静默无效，仍是 4 行，实际 {n_wrong}"
assert n_right == 7, f"正确拆分后应为 7 行，实际 {n_right}"
assert right.tolist() == ["渗油", "异物", "放电,发热", "无", "发热", "放电", "无"], \
    f"拆分结果不对：{right.tolist()}"
assert right.index.tolist() == [0, 0, 1, 2, 3, 3, 3], f"索引应为重复索引，实际 {right.index.tolist()}"
assert right.index.has_duplicates, "explode 后索引必然重复"
print("TODO(9) 通过 -> 直接 explode", n_wrong, "行（无效）| 先 split 再 explode",
      n_right, "行 | 索引", right.index.tolist())

In [ ]:
# TODO(10)：多分隔符与 ignore_index：
#           '放电,发热' 用 split('/') 与 split('[/,]', regex=True) 的结果分别存入 s_slash / s_regex
#           split_demo.explode("缺陷类型", ignore_index=True) 的索引存入 idx_ignore
#           含 NaN 的列 split 后再 explode 的行数存入 n_nan_case
# 提示：pd.Series(["放电,发热"]).str.split("/") ；str.split(r"[/,]", regex=True)
#        pd.DataFrame({"编号":[1,2],"缺陷类型":["渗油",None]})["缺陷类型"].str.split("/").explode()
s_slash = ____
s_regex = ____
idx_ignore = ____
nan_case = ____
n_nan_case = ____
# ---- 验收 ----
assert s_slash == [["放电,发热"]], f"单字符分隔符切不开，实际 {s_slash}"
assert s_regex == [["放电", "发热"]], f"正则可切多个分隔符，实际 {s_regex}"
assert idx_ignore == [0, 1, 2, 3, 4, 5, 6], f"ignore_index 应重排为 0..6，实际 {idx_ignore}"
assert n_nan_case == 2, f"NaN 会保留为一行，行数应为 2，实际 {n_nan_case}"
assert pd.Series(["无"]).str.split("/").tolist() == [["无"]], "无分隔符时返回单元素列表"
print("TODO(10) 通过 -> 正则切分", s_regex, "| ignore_index", idx_ignore,
      "| NaN 行数", n_nan_case)

---

## 综合题

### 综合 ① 长 ↔ 宽 闭环与行数对账

要求证明：`dev → long → wide → long_back` 的每一步都能说清"行数为什么是这个数"。

In [ ]:
# TODO(11)：综合①：产出闭环的四个阶段数字，存入 closure 字典：
#          dev 行数 / long 组合数 / wide 满格数 / long_back 行数
#          以及「dropna 后回到 long 的行数」→ 键名 "回到组合数"
# 提示：满格数 = wide.shape[0] * wide.shape[1] ；dropna(subset=["平均负荷"])
closure = ____
# ---- 验收 ----
assert closure["dev 行数"] == 270, f"明细应为 270 行，实际 {closure['dev 行数']}"
assert closure["long 组合数"] == 88, f"组合应为 88，实际 {closure['long 组合数']}"
assert closure["wide 满格数"] == 90, f"满格数应为 90，实际 {closure['wide 满格数']}"
assert closure["long_back 行数"] == 90, f"melt 应带出满格 90 行，实际 {closure['long_back 行数']}"
assert closure["long 有值组合数"] == 87, f"88 个组合里有 1 个值全缺，实际 {closure['long 有值组合数']}"
assert closure["有值格子数"] == 87, f"90 格减 3 个空格应为 87，实际 {closure['有值格子数']}"
assert closure["wide 满格数"] - closure["long 组合数"] == 2, "90 - 88 = 2 个组合不存在的格子"
assert closure["long 组合数"] - closure["long 有值组合数"] == 1, "88 - 87 = 1 个值全缺的组合"
assert closure["long_back 行数"] != closure["long 组合数"], \
    "关键结论：长→宽→长 行数【不守恒】，90（满格）≠ 88（组合）"
# 值层面能否原路返回（两边都只看有值的）
back = long_back.dropna(subset=["平均负荷"]).set_index([ROW_KEY, COL_KEY])
assert set(back.index) == set(long.dropna().index), "有值组合的索引集合应一致"
assert bool(np.allclose(back["平均负荷"].sort_index(), long.dropna().sort_index())), "值应完全一致"
print("综合① 通过 ->", closure)
print("   关键：90（满格）≠ 88（组合）≠ 87（有值），reshape 闭环三个口径都要对账")

### 综合 ② 一张可交付的交叉报表

要求：以「台区编号 × 设备类型」为轴，值为平均负荷；
用 `assign` 加一列"台区平均"，缺格填 0，最后按"台区平均"降序排列。

In [ ]:
# TODO(12)：综合②：产出报表 report：
#           ① 对 dev 做 pivot_table(index=ROW_KEY, columns=COL_KEY, values=负荷值,
#              aggfunc="mean", fill_value=0)
#           ② 加一列 "台区平均" = 该行 5 个设备类型的均值
#           ③ 按 "台区平均" 降序排序
#          此外把「填 0 的格子数」存入 n_zero、行数存入 n_rows、列数存入 n_cols
# 提示：pt = dev.pivot_table(index=ROW_KEY, columns=COL_KEY, values=VALUE_COL,
#                             aggfunc="mean", fill_value=0)
#        report = pt.assign(台区平均=pt.mean(axis=1)).sort_values("台区平均", ascending=False)
pt = ____
report = ____
n_zero = ____
n_rows, n_cols = ____
# ---- 验收 ----
assert n_rows == 18, f"报表应有 18 行（每个台区一行），实际 {n_rows}"
assert n_cols == 6, f"报表应有 6 列（5 设备类型 + 台区平均），实际 {n_cols}"
assert "台区平均" in report.columns, "应有 台区平均 列"
assert list(report.columns[-1:]) == ["台区平均"], f"台区平均应在最后一列，实际 {list(report.columns)}"
assert set(report.columns[:-1]) == set(pt.columns), "前 5 列应恰好是 5 个设备类型"
# 填 0 的格子：2 个组合不存在 + 1 个值全缺（也被填成 0）→ 共 3 个
assert n_zero == 3, f"填 0 的格子应为 3 个，实际 {n_zero}"
assert report["台区平均"].is_monotonic_decreasing, "应按台区平均降序"
assert np.allclose(report[list(pt.columns)].mean(axis=1).to_numpy(),
                   report["台区平均"].to_numpy()), "台区平均应等于 5 个设备类型的均值"
assert int(report[list(pt.columns)].isna().sum().sum()) == 0, "fill_value=0 后不应有 NaN"
print("综合② 通过 -> 报表", report.shape, "| 填 0 格子", n_zero, "个")
print("   注意：那 1 个「值全缺」的格子也被填成了 0 —— 交付前应在报告里说明")
print(report.round(2).head(5).to_string())

> ⚠️ **这道题的隐藏结论**：`fill_value=0` 把 **3** 个格子都填成了 `0`，
> 但其中只有 **2** 个是"组合不存在"（填 0 合理），
> 剩下 **1** 个是"`STATION_A_01 × 变压器` 有设备但负荷值全缺"（填 0 是把缺失当 0）。
>
> 竞赛里这属于"结果表能跑但语义有瑕疵"。
> 规范做法是：**先 `fill_value` 诊断出的那一格单独处理**（置 `NaN` 并在报告中说明），
> 其余 2 格再填 0。

---

### 复盘提问

**复盘问自己**（六题都能答上来才算过）：

1. 宽表里的 `NaN` 有哪两种成因？用什么参数能区分它们？
2. 为什么 `pivot` 会因重复索引报错，而 `pivot_table` 不会？
3. `pivot_table` 与 `groupby + unstack` 是什么关系？`aggfunc` 默认值是什么？
4. `stack` / `unstack` 在 pandas 3.0 有哪两处破坏性变更？
5. `pd.crosstab(index, columns, values=..., aggfunc=...)` 和纯计数版有什么区别？
6. `explode` 前为什么必须先 `str.split`？多分隔符怎么写？

答不上来的，回 `_notes/错题本.md` 记一笔。